# Week 07 · Topic and Semantic Analysis
**ISB02306504 – Social Media Mining** · Information Systems for Business · Odd Semester 2026/2027

**81k Google Play reviews of BRImo**, the mobile banking app of Bank Rakyat Indonesia (July 2025 – June 2026).

Sentiment tells a bank *how many* users are unhappy. This lab answers the next question: **unhappy about what?** (slide 3).

| Part | What we do | Slides |
|---|---|---|
| 0–2 | Setup, load, EDA, preprocessing for topic analysis | – |
| 3 | Keywords: term frequency, TF-IDF, segment comparison, phrases, co-occurrence networks | 5–14 |
| 4 | Topic modeling with LDA: fitting, choosing K, coherence, interpretation, prevalence over time | 16–25 |
| 5 | Semantic representations: word embeddings, cosine similarity, sentence embeddings, semantic search, clustering | 27–35 |
| 6 | Transformers: contextual vectors (the same word, different meanings) | 37–39 |
| 7 | Which representation to use, and turning themes into actions | 40–41 |
| 8 | Class activity and homework | – |

**Runtime.** *Runtime → Change runtime type → T4 GPU*.

---
## 0. Setup

In [ ]:
import re, time, math, unicodedata, urllib.request, warnings, importlib.util, subprocess, sys
from pathlib import Path
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from scipy import sparse
from scipy.optimize import linear_sum_assignment
from IPython.display import display

import nltk
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, TruncatedSVD, PCA
from sklearn.preprocessing import normalize, Normalizer
from sklearn.pipeline import make_pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_mutual_info_score
from sklearn.metrics.pairwise import cosine_similarity
import sklearn

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 11, "font.size": 9})
SENT_COLORS = {"Negative": "#eb6834", "Neutral": "#8a8984", "Positive": "#2a78d6"}
TOPIC_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948",
                "#8a8984", "#5b3a29", "#17becf", "#bcbd22"]
TIMINGS = {}          # we record how long each representation takes, for the comparison in part 7
print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

Part 5 uses `sentence-transformers`, which Colab normally has preinstalled. The cell below installs it only if missing. If it still cannot be loaded (for example, no internet), the notebook falls back to a classic **LSA** representation, so every cell still runs.

In [ ]:
if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=False)
try:
    import torch
    from sentence_transformers import SentenceTransformer
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    ST_AVAILABLE = True
except Exception as e:
    DEVICE, ST_AVAILABLE = "cpu", False
    print("sentence-transformers not available:", type(e).__name__)
print("sentence-transformers available:", ST_AVAILABLE, "| device:", DEVICE)

In [ ]:
DATA_URL = ("https://raw.githubusercontent.com/satriaadi9/smm/refs/heads/main/"
            "w7%20-%20topic%20%26%20semantic%20analysis/bri-apps-review.csv")
DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)

def download_once(url, dest):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"{dest.name}: using cached copy ({dest.stat().st_size / 1e6:.1f} MB)")
        return dest
    tmp = dest.with_name(dest.name + ".part")
    t0 = time.perf_counter()
    urllib.request.urlretrieve(url, tmp)
    tmp.rename(dest)
    print(f"{dest.name}: downloaded {dest.stat().st_size / 1e6:.1f} MB in {time.perf_counter() - t0:.0f}s")
    return dest

PATH = download_once(DATA_URL, DATA_DIR / "bri-apps-review.csv")

---
## 1. Load and explore
### 1.1 Peek at the raw file first

In [ ]:
with open(PATH, encoding="utf-8") as f:
    for _ in range(2):
        print(repr(f.readline()[:160]))

Two details we would miss with a blind `pd.read_csv(url)`:
- the separator is a **semicolon**, not a comma (review texts contain commas);
- the file starts with a **byte-order mark** (`﻿`), which would end up glued to the first column name. `encoding="utf-8-sig"` removes it.

In [ ]:
try:
    pd.read_csv(PATH, nrows=5)
    print("default read worked?!")
except Exception as e:
    print("Default read_csv fails:", type(e).__name__, "-", str(e)[:90])

COLS = {"App ID": "app_id", "Nama Aplikasi": "app", "Reviewer": "reviewer", "Rating": "rating", "Ulasan": "review",
        "Tanggal": "date", "Likes": "likes", "Balasan Dev": "dev_reply", "Versi App": "version_raw", "Sentimen": "sentiment"}
df = pd.read_csv(PATH, sep=";", encoding="utf-8-sig").rename(columns=COLS)
df["date"] = pd.to_datetime(df["date"], format="%d/%m/%y %H.%M")
df["sentiment"] = df["sentiment"].map({"Positif": "Positive", "Negatif": "Negative", "Netral": "Neutral"})
df["month"] = df["date"].dt.to_period("M").dt.to_timestamp()
print(df.shape)
display(df.head(3))

| Column | Meaning | Use |
|---|---|---|
| `review` | the user's review text | **main input** |
| `rating` | star rating 1–5 | segmenting |
| `sentiment` | Positive / Negative / Neutral label provided with the data | segmenting |
| `date`, `likes` | when, and how many users found the review helpful | trends, weighting |
| `dev_reply` | BRI's reply to the review | **not** user voice (see 1.3) |
| `version_raw` | app version | needs repair (see 1.3) |
| `app_id`, `app`, `reviewer` | constant / anonymised IDs | no |

### 1.2 Overview

In [ ]:
print("Missing values:", df.isna().sum()[lambda s: s > 0].to_dict())
print("Apps:", df["app"].unique().tolist(), "| Date range:", df["date"].min().date(), "→", df["date"].max().date())

fig, axes = plt.subplots(1, 2, figsize=(12, 3.3), gridspec_kw={"width_ratios": [1, 2]})
counts = df["sentiment"].value_counts().reindex(SENT_COLORS)
axes[0].bar(counts.index, counts.values, color=[SENT_COLORS[k] for k in counts.index], width=0.6)
for i, v in enumerate(counts.values):
    axes[0].text(i, v, f"{v:,}\n({v / len(df):.1%})", ha="center", va="bottom", fontsize=8)
axes[0].set_ylim(0, counts.max() * 1.25); axes[0].set_title("Reviews per sentiment")

monthly = pd.crosstab(df["month"], df["sentiment"]).reindex(columns=SENT_COLORS)
for lab, c in SENT_COLORS.items():
    axes[1].plot(monthly.index, monthly[lab], marker="o", color=c, label=lab, lw=2)
axes[1].set_title("Reviews per month"); axes[1].legend()
plt.tight_layout(); plt.show()
print("Negative share per month:", {f"{m:%b %y}": f"{v:.0%}" for m, v in (monthly["Negative"] / monthly.sum(axis=1)).items()})

About a quarter of reviews are negative. June 2026 is a partial month (the data ends on the 19th), so its lower volume is not a real drop. That kind of detail matters when someone asks "are complaints going down?".

### 1.3 Data quality checks
**Labels.** Where does `sentiment` come from?

In [ ]:
display(pd.crosstab(df["rating"], df["sentiment"]).reindex(columns=SENT_COLORS))
display(df[(df["rating"] == 5) & (df["sentiment"] == "Negative")].sample(5, random_state=SEED)[["rating", "sentiment", "review"]])

Unlike last week's dataset, the label is **not** a simple star rule: thousands of 5-star reviews are labelled Negative, and many of them really are complaints ("tolong diperbaiki…"). The label was probably produced by a text classifier or lexicon. We use it to *segment* reviews, and we treat it as useful but imperfect.

In [ ]:
print(df["version_raw"].value_counts().head(6).to_dict())

num = pd.to_numeric(df["version_raw"], errors="coerce")
minutes = (num * 24 * 60).round()
from_fraction = ((minutes // 60).astype("Int64").astype("string") + "."
                 + (minutes % 60).astype("Int64").astype("string").str.zfill(2))
txt = df["version_raw"].astype("string").str.extract(r"^(\d{1,2})\.(\d{2})\.\d{2}$")
from_text = txt[0].str.lstrip("0") + "." + txt[1]
df["version"] = from_fraction.fillna(from_text)

check = df.groupby("version")["date"].agg(["count", "median"]).query("count > 1000").sort_values("median")
display(check)

A version such as `3.25` was read as the *time* 03:25 and stored as a fraction of a day (`0.1424`). Multiplying by 24 × 60 minutes recovers it. The sanity check confirms the repair: higher versions appear later in time. Always open suspicious columns before trusting them.

**Developer replies.** Every review has a reply from BRI:

In [ ]:
print(f"Reviews with a developer reply: {df['dev_reply'].notna().mean():.0%}")
template = df["dev_reply"].str.replace(r"Tks~.*$", "", regex=True).str.strip()
print(f"Distinct reply texts: {df['dev_reply'].nunique():,} | distinct after removing the agent signature: {template.nunique():,}")
display(template.value_counts().head(4).to_frame("replies"))

The replies are templates signed by agents: the four most common texts alone cover about half of all replies. If we mixed them into the corpus, "sobat bri" and "mohon maaf" would become the biggest "topic". They are the company's voice, not the users', so we leave them out.

**Duplicates and length.**

In [ ]:
df["n_words"] = df["review"].str.split().str.len()
print(f"Exact duplicate reviews: {df['review'].str.lower().str.strip().duplicated().mean():.1%}")
print(f"Reviews with ≤ 3 words: {(df['n_words'] <= 3).mean():.1%}")
display(df["review"].str.lower().str.strip().value_counts().head(8).to_frame("copies"))
print(df.groupby("sentiment")["n_words"].describe()[["mean", "50%", "75%", "max"]].round(1))

Many reviews are one or two words of praise ("mantap", "bagus"). They tell us the user is happy, but they contain no *theme*. Negative reviews are much longer, because people explain what went wrong. For topic analysis this means:
- most of the thematic signal is in the **negative** reviews, which is also what the bank needs to act on;
- very short texts should be excluded from topic modeling (slide 23: *short social posts are sparse*).

---
## 2. Preprocessing for topic analysis
Topic and keyword analysis need a different pipeline from last week's classifier. There, `tidak` was precious; here, we want **content words** that describe *what* the review is about.

| Step | Decision | Why (slide 5) |
|---|---|---|
| NFKC, lowercase, elongation, slang | normalize | merge spelling variants |
| `-nya` suffix | strip when the base word is common | `saldonya` = `saldo` (morphological variation) |
| Generic + informal stopwords | remove | *generic vocabulary* |
| `bri`, `brimo`, `aplikasi`, `bank` | remove | *brand dominance*, *platform vocabulary* |
| Request words `tolong`, `mohon`, `perbaiki` | remove | they appear in almost every complaint and describe no theme |
| Frequent word pairs | join into phrases (`layar_putih`) | *phrase loss* |
| Reviews with < 3 content words | exclude from topic models | too sparse |

In [ ]:
SLANG = {"gk": "tidak", "ga": "tidak", "gak": "tidak", "nggak": "tidak", "tdk": "tidak", "yg": "yang", "bgt": "banget",
         "udh": "sudah", "udah": "sudah", "sdh": "sudah", "tp": "tapi", "aja": "saja", "dr": "dari", "utk": "untuk",
         "dgn": "dengan", "krn": "karena", "jd": "jadi", "bs": "bisa", "blm": "belum", "lg": "lagi", "sy": "saya",
         "gmn": "gimana", "knp": "kenapa", "trs": "terus", "mulu": "terus", "apk": "aplikasi", "app": "aplikasi",
         # banking terms and spelling variants (several were found with the word embeddings in part 5)
         "tf": "transfer", "trf": "transfer", "tranfer": "transfer", "trasfer": "transfer", "eror": "error",
         "erorr": "error", "verivikasi": "verifikasi", "vermuk": "verifikasi wajah", "muka": "wajah",
         "poto": "foto", "selfi": "selfie", "kepotong": "terpotong", "lemot": "lambat", "lelet": "lambat",
         "gabisa": "tidak bisa", "gbs": "tidak bisa", "gakbisa": "tidak bisa", "tlg": "tolong", "mslh": "masalah",
         "skrg": "sekarang", "skrng": "sekarang", "pdhl": "padahal", "pdhal": "padahal", "ksi": "kasih",
         "pasword": "password", "sandi": "password", "membatu": "membantu", "sanggat": "sangat", "ngga": "tidak",
         "nggk": "tidak", "ngk": "tidak", "dah": "sudah", "kyk": "kayak", "mlh": "malah", "msh": "masih", "gt": "gitu",
         "jg": "juga", "knapa": "kenapa", "trus": "terus", "bukak": "buka", "loging": "login"}
URL_RE = re.compile(r"https?://\S+|www\.\S+")

def normalize_tokens(text):
    s = unicodedata.normalize("NFKC", str(text)).lower()
    s = URL_RE.sub(" ", s)
    s = re.sub(r"([a-z])\1{2,}", r"\1", s)                 # elongation
    out = []
    for t in re.findall(r"[a-z]+", s):                      # letters only: numbers carry no theme here
        out.extend(SLANG.get(t, t).split())
    return out

t0 = time.perf_counter()
df["tokens_all"] = df["review"].map(normalize_tokens)
print(f"Tokenised {len(df):,} reviews in {time.perf_counter() - t0:.1f}s")
print(df.loc[1, "review"], "\n→", df.loc[1, "tokens_all"])

In [ ]:
# Data-driven "-nya" handling: strip the suffix only if the base word is itself common in this corpus
freq_all = Counter(t for toks in df["tokens_all"] for t in toks)
NYA_MAP = {w: w[:-3] for w in freq_all if w.endswith("nya") and len(w) > 6 and freq_all[w[:-3]] >= 20}
print(len(NYA_MAP), "-nya forms mapped, e.g.", dict(list(sorted(NYA_MAP.items(), key=lambda kv: -freq_all[kv[0]]))[:8]))

SW_GENERIC = set(stopwords.words("indonesian"))
SW_INFORMAL = {"nya", "sih", "deh", "dong", "kok", "ya", "yah", "lah", "nih", "aja", "saja", "banget", "gitu", "udah",
               "kalo", "pake", "gimana", "kenapa", "terus", "tapi", "gak", "aku", "gue", "gua", "kak", "min", "pak",
               "bu", "bang", "ok", "oke", "sangat", "sekali", "kali", "jadi", "sama", "lagi", "mau", "bisa", "tidak"}
SW_DOMAIN = {"bri", "brimo", "aplikasi", "bank", "bri mo", "mobile"}
SW_REQUEST = {"tolong", "mohon", "perbaiki", "diperbaiki", "segera", "kasih", "bintang", "terima", "makasih", "solusi", "coba"}
# Evaluative words say HOW users feel, not WHAT about. Sentiment already captures them; for topics they blur every theme.
SW_EVAL = {"bagus", "jelek", "susah", "ribet", "bikin", "kecewa", "mudah", "parah", "payah", "buruk", "mantap", "baik",
           "pas", "sih", "gimana", "mending", "aneh", "ganggu", "kesel"}
STOP = SW_GENERIC | SW_INFORMAL | SW_DOMAIN | SW_REQUEST | SW_EVAL

def is_content(t):
    return len(t) > 2 and t not in STOP

def content_tokens(tokens):
    return [t for t in (NYA_MAP.get(t, t) for t in tokens) if is_content(t)]

df["tokens"] = df["tokens_all"].map(content_tokens)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, col, title, color in [(axes[0], "tokens_all", "Before: all tokens", "#8a8984"),
                              (axes[1], "tokens", "After: content tokens", "#2a78d6")]:
    top = Counter(t for toks in df.loc[df["sentiment"] == "Negative", col] for t in toks).most_common(20)[::-1]
    ax.barh([w for w, _ in top], [c for _, c in top], color=color)
    ax.set_title(f"{title} (negative reviews)"); ax.set_xlabel("frequency")
plt.tight_layout(); plt.show()

On the left, the top of the list is function words, the brand and the app name. On the right, the words describe actual problems: *login, saldo, layar, verifikasi, update*. (We also removed evaluative words such as *bagus, susah, ribet*: they tell us *how* users feel, which sentiment already captures, not *what* they talk about.)

### Phrases
"Customer service" means more than "customer" + "service" (slide 9). We find word pairs that occur together much more often than chance, using **normalized PMI** (NPMI, between −1 and 1), and join them into single tokens.

In [ ]:
# only pairs that are truly adjacent in the review, and both words are content words
mapped = df["tokens_all"].map(lambda toks: [NYA_MAP.get(t, t) for t in toks])
unigram = Counter(t for toks in mapped for t in toks if is_content(t))
bigram = Counter((a, b) for toks in mapped for a, b in zip(toks, toks[1:]) if is_content(a) and is_content(b))
N_uni = sum(unigram.values())

def npmi(pair, count):
    p_ab = count / N_uni
    pmi = math.log(p_ab / ((unigram[pair[0]] / N_uni) * (unigram[pair[1]] / N_uni)))
    return pmi / -math.log(p_ab)

phr = pd.DataFrame([(f"{a} {b}", c, npmi((a, b), c)) for (a, b), c in bigram.items() if c >= 30],
                   columns=["phrase", "count", "npmi"]).sort_values("npmi", ascending=False)
PHRASES = dict(zip(phr["phrase"], phr["npmi"]))
PHRASES = {p: s for p, s in PHRASES.items() if s >= 0.45}
print(f"{len(PHRASES)} phrases kept (count ≥ 30, NPMI ≥ 0.45)")
display(phr.head(15).round(2))

def to_content(tokens):
    """Normalised tokens → content tokens, with known phrases joined (layar putih → layar_putih)."""
    toks = [NYA_MAP.get(t, t) for t in tokens]
    out, i = [], 0
    while i < len(toks):
        here = PHRASES.get(f"{toks[i]} {toks[i + 1]}", 0) if i + 1 < len(toks) else 0
        nxt = PHRASES.get(f"{toks[i + 1]} {toks[i + 2]}", 0) if i + 2 < len(toks) else 0
        if here and here >= nxt:                        # "muncul layar putih": prefer the stronger pair
            out.append(f"{toks[i]}_{toks[i + 1]}"); i += 2
        else:
            if is_content(toks[i]):
                out.append(toks[i])
            i += 1
    return out

df["tokens"] = df["tokens_all"].map(to_content)
df["n_tokens"] = df["tokens"].str.len()
print(df.loc[df["n_tokens"] > 6, ["review", "tokens"]].sample(3, random_state=1).to_string())

Finally, the **modeling corpus**. Topic models learn from co-occurrence, so 3,000 copies of the same short review add weight without adding information. We fit models on **unique, informative** texts (at least 3 content tokens) and later apply them to *all* reviews to measure prevalence.

In [ ]:
df["doc"] = df["tokens"].str.join(" ")
informative = df["n_tokens"] >= 3
print(f"Informative reviews (≥ 3 content tokens): {informative.mean():.1%} of all, "
      f"{informative[df['sentiment'] == 'Negative'].mean():.1%} of negative")
display(pd.crosstab(df["sentiment"], informative, normalize="index").rename(columns={True: "informative", False: "too short"}).round(3))

neg = (df[informative & (df["sentiment"] == "Negative")]
       .drop_duplicates("doc").reset_index(drop=True))
print(f"Negative modeling corpus: {len(neg):,} unique informative reviews")

---
## 3. Keywords and co-occurrence  *(slides 4–14)*
Three levels of analysis (slide 4): **keywords** (which terms stand out), **topics** (which words form themes) and **semantics** (which texts mean the same thing). We start with keywords: fast, interpretable, but purely lexical.

### 3.1 TF-IDF: frequent here, uncommon elsewhere
Raw frequency answers "what is common?". TF-IDF answers "what is *characteristic* of this document?". Document-level keywords for three negative reviews:

In [ ]:
t0 = time.perf_counter()
tfidf = TfidfVectorizer(token_pattern=r"\S+", min_df=5, sublinear_tf=True)
X_tfidf = tfidf.fit_transform(df.loc[informative, "doc"])
TIMINGS["TF-IDF"] = time.perf_counter() - t0
vocab_tfidf = tfidf.get_feature_names_out()
row_of = {idx: i for i, idx in enumerate(df.index[informative])}

examples = df[informative & (df["sentiment"] == "Negative") & df["n_tokens"].between(8, 25)].sample(3, random_state=3)
for idx, r in examples.iterrows():
    row = X_tfidf[row_of[idx]].toarray().ravel()
    top = [vocab_tfidf[j] for j in row.argsort()[::-1][:5] if row[j] > 0]
    print(f"• {r['review'][:150]}\n  keywords → {' • '.join(top)}\n")

Abbreviations and typos that slip through the slang dictionary often rank high, because TF-IDF rewards words that are rare in the corpus. Each one you spot is a candidate for the dictionary.

🧑‍🏫 **Slide 10 questions, on real reviews:** which keywords are generic and which are useful to BRI? Would a phrase be more useful than its single words? Which preprocessing decision (stopwords, `-nya`, phrases) would change the ranking?

### 3.2 Corpus-level: frequency vs TF-IDF

In [ ]:
neg_mask = (df.loc[informative, "sentiment"] == "Negative").values
freq_rank = Counter(t for toks in df.loc[informative & (df["sentiment"] == "Negative"), "tokens"] for t in toks)
mean_tfidf = pd.Series(np.asarray(X_tfidf[neg_mask].mean(axis=0)).ravel(), index=vocab_tfidf)
display(pd.DataFrame({"by frequency": [w for w, _ in freq_rank.most_common(15)],
                      "by mean TF-IDF": mean_tfidf.nlargest(15).index}))

The two lists overlap a lot at corpus level, because our stopword work already removed most generic words. TF-IDF pushes up words that dominate *short* documents (sublinear TF and length normalisation), so it is most useful for document-level keywords.

### 3.3 Segment comparison: what characterises negative vs positive reviews?
Stakeholder value usually comes from **comparisons** (slide 8). A robust way to compare two groups is the **weighted log-odds ratio with an informative prior** (Monroe et al., 2008). It is like comparing frequencies, but rare words do not get extreme scores, and the result is a *z*-score.

In [ ]:
def log_odds_z(counts_a, counts_b, prior_scale=0.01, min_count=20):
    """Weighted log-odds (Monroe et al. 2008). Positive z → typical of A, negative → typical of B."""
    vocab = [w for w in set(counts_a) | set(counts_b) if counts_a[w] + counts_b[w] >= min_count]
    bg = Counter({w: counts_a[w] + counts_b[w] for w in vocab})
    a0 = prior_scale * sum(bg.values())
    alpha = {w: a0 * bg[w] / sum(bg.values()) for w in vocab}
    n_a, n_b = sum(counts_a[w] for w in vocab), sum(counts_b[w] for w in vocab)
    z = {}
    for w in vocab:
        la = math.log((counts_a[w] + alpha[w]) / (n_a + a0 - counts_a[w] - alpha[w]))
        lb = math.log((counts_b[w] + alpha[w]) / (n_b + a0 - counts_b[w] - alpha[w]))
        z[w] = (la - lb) / math.sqrt(1 / (counts_a[w] + alpha[w]) + 1 / (counts_b[w] + alpha[w]))
    return pd.Series(z).sort_values()

c_neg = Counter(t for toks in df.loc[df["sentiment"] == "Negative", "tokens"] for t in toks)
c_pos = Counter(t for toks in df.loc[df["sentiment"] == "Positive", "tokens"] for t in toks)
z = log_odds_z(c_neg, c_pos)
top = pd.concat([z.head(15), z.tail(15)])

fig, ax = plt.subplots(figsize=(7, 6.5))
ax.barh(top.index, top.values, color=[SENT_COLORS["Positive"] if v < 0 else SENT_COLORS["Negative"] for v in top.values])
ax.axvline(0, color="#8a8984", lw=0.8)
ax.set_xlabel("z-score  (← positive reviews | negative reviews →)")
ax.set_title("Words that characterise negative vs positive reviews")
plt.tight_layout(); plt.show()

Negative reviews are characterised by **access and failure** words (buka, login, error, gagal, saldo, layar_putih); positive reviews by **help, ease and speed** (membantu, cepat, memudahkan, praktis, aman). Note `transaksi` on the positive side: happy users praise smooth transactions, unhappy users name the specific failure instead. That is already a useful message for a product owner.

### 3.4 Unigrams vs phrases  *(slide 9)*

In [ ]:
sentence = "customer service lambat sekali tapi proses refund mudah"
for ngram in [(1, 1), (2, 2)]:
    print(ngram, list(CountVectorizer(ngram_range=ngram).fit([sentence]).get_feature_names_out()))

phrase_counts = Counter(t for toks in neg["tokens"] for t in toks if "_" in t)
print("\nMost frequent phrases in negative reviews:", [p for p, _ in phrase_counts.most_common(15)])

`layar_putih` (blank white screen), `verifikasi_wajah` (face verification) and `kode_otp` are business concepts; their single words are much less clear on their own.

### 3.5 Co-occurrence: which words appear together?  *(slides 11–13)*
First decide the **window**: here, *the same review*. Each cell of the matrix counts how many negative reviews contain both words.

In [ ]:
cv_bin = CountVectorizer(token_pattern=r"\S+", min_df=20, binary=True)
B = cv_bin.fit_transform(neg["doc"])
terms = cv_bin.get_feature_names_out()
co = (B.T @ B).toarray()
df_counts = np.diag(co).copy()

top_terms = [t for t, _ in Counter(dict(zip(terms, df_counts))).most_common(12)]
ti = [list(terms).index(t) for t in top_terms]
block = co[np.ix_(ti, ti)].copy()
np.fill_diagonal(block, 0)                      # a word with itself is not interesting
co_top = pd.DataFrame(block, index=top_terms, columns=top_terms)
display(co_top.style.background_gradient(cmap="Blues", axis=None))

**Caution (slide 12):** raw counts favour frequent words. `login` co-occurs with everything simply because it is everywhere. An association measure such as **NPMI** asks whether two words appear together *more than expected by chance*:

In [ ]:
N_docs = B.shape[0]
iu = np.triu_indices(len(terms), k=1)
pairs = pd.DataFrame({"a": terms[iu[0]], "b": terms[iu[1]], "count": co[iu]})
pairs = pairs[pairs["count"] >= 40].copy()
p_a = df_counts[iu[0]] / N_docs; p_b = df_counts[iu[1]] / N_docs
p_ab = co[iu] / N_docs
with np.errstate(divide="ignore", invalid="ignore"):
    npmi_all = np.log(p_ab / (p_a * p_b)) / -np.log(p_ab)
pairs["npmi"] = npmi_all[pairs.index]
display(pd.concat({"top pairs by raw count": pairs.nlargest(10, "count")[["a", "b", "count"]].reset_index(drop=True),
                   "top pairs by NPMI (count ≥ 40)": pairs.nlargest(10, "npmi")[["a", "b", "npmi"]].round(2).reset_index(drop=True)}, axis=1))

The raw-count list is dominated by the most frequent words. The NPMI list surfaces tight, specific pairs. Very high NPMI pairs are often fixed expressions (*play store*) or **text copied from an error message** that many users quote: worth reading, because a quoted error message is itself a strong signal of one specific problem. For a word network we combine both: enough support (count) **and** real association (NPMI).

In [ ]:
top40 = set(t for t, _ in Counter(dict(zip(terms, df_counts))).most_common(45))
edges = pairs[(pairs["a"].isin(top40)) & (pairs["b"].isin(top40)) & (pairs["npmi"] >= 0.18) & (pairs["count"] >= 50)]
G = nx.Graph()
for _, e in edges.iterrows():
    G.add_edge(e["a"], e["b"], weight=e["count"], npmi=e["npmi"])
communities = nx.community.greedy_modularity_communities(G, weight="npmi")
comm_of = {n: i for i, c in enumerate(communities) for n in c}

fig, ax = plt.subplots(figsize=(10, 7))
pos = nx.spring_layout(G, seed=SEED, k=1.4, weight="npmi", iterations=200)
freq_neg = dict(zip(terms, df_counts))
nx.draw_networkx_edges(G, pos, ax=ax, width=[0.6 + 3 * G[u][v]["npmi"] for u, v in G.edges()], edge_color="#bdbcb8")
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=[40 + freq_neg[n] / 4 for n in G],
                       node_color=[TOPIC_COLORS[comm_of[n] % len(TOPIC_COLORS)] for n in G], alpha=0.9)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8)
ax.set_title("Word co-occurrence network, negative reviews (colour = community, edge width = NPMI)"); ax.axis("off")
plt.tight_layout(); plt.show()
for i, c in enumerate(communities):
    print(f"community {i}: {', '.join(sorted(c, key=lambda n: -freq_neg[n]))}")

Look for clusters, bridges and strong pairings (slide 13). Communities often already resemble themes: verification documents, network problems, money and transactions, app not opening. Remember that this is a **lexical** network between words, not a social network between users.

### 3.6 The limit of lexical analysis  *(slide 14)*

In [ ]:
PAIRS = [
    ("saldo terpotong tapi transfer gagal", "uang sudah keluar dari rekening namun tidak sampai ke penerima"),
    ("aplikasi tidak bisa dibuka, layar putih terus", "setiap kali dibuka cuma blank putih"),
    ("cs tidak pernah membalas keluhan saya", "customer service mengabaikan laporan saya"),
    ("saldo terpotong tapi transfer gagal", "tampilan aplikasinya bagus dan warnanya cantik"),
]
def jaccard(a, b):
    a, b = set(content_tokens(normalize_tokens(a))), set(content_tokens(normalize_tokens(b)))
    return len(a & b) / len(a | b) if a | b else 0.0

lexical = pd.DataFrame(PAIRS, columns=["text A", "text B"])
lexical["shared content words"] = [jaccard(a, b) for a, b in PAIRS]
tf_pairs = tfidf.transform([" ".join(to_content(normalize_tokens(t))) for p in PAIRS for t in p])
lexical["TF-IDF cosine"] = [cosine_similarity(tf_pairs[2 * i], tf_pairs[2 * i + 1])[0, 0] for i in range(len(PAIRS))]
display(lexical.round(2))

The first three pairs describe the same problem with almost no shared words, so lexical methods score them close to zero, just like the truly unrelated last pair. This is the motivation for embeddings in part 5.

---
## 4. Topic modeling with LDA  *(slides 16–25)*
When thousands of reviews cannot be read one by one, a topic model gives a compact thematic summary (slide 16). In **LDA**, *documents are mixtures of topics and topics are mixtures of words* (slides 17–19).

We model the **negative** reviews, because that is where the bank needs to know "about what?".

### 4.1 Vectorize and fit (slide 25 workflow)
LDA expects **raw counts** (not TF-IDF). The vocabulary should be neither too rare (`min_df`) nor too generic (`max_df`). Our phrases are already joined, so we use single tokens instead of `ngram_range=(1, 2)`; otherwise a phrase and its words would be counted twice.

In [ ]:
vectorizer = CountVectorizer(token_pattern=r"\S+", min_df=5, max_df=0.90)
X = vectorizer.fit_transform(neg["doc"])
vocab = vectorizer.get_feature_names_out()
print(f"Document-term matrix: {X.shape[0]:,} reviews × {X.shape[1]:,} terms")

t0 = time.perf_counter()
lda5 = LatentDirichletAllocation(n_components=5, random_state=SEED, learning_method="batch", max_iter=20).fit(X)
TIMINGS["LDA (K=5)"] = time.perf_counter() - t0
print(f"Fitted in {TIMINGS['LDA (K=5)']:.0f}s")

def topic_table(model, n=10):
    weights = model.components_ / model.components_.sum(axis=1, keepdims=True)    # P(word | topic)
    return pd.DataFrame({f"Topic {k}": [f"{vocab[j]} ({weights[k, j]:.3f})" for j in weights[k].argsort()[::-1][:n]]
                         for k in range(model.n_components)})
display(topic_table(lda5))

Each column is a **topic → word** distribution: the numbers are P(word | topic). Read all ten words together; never name a topic from one keyword (slide 20).

Now the **document → topic** distribution for one review (slide 20, right side):

In [ ]:
doc_topic5 = lda5.transform(X)
i = int(np.argmax((doc_topic5.max(axis=1) < 0.6) & (neg["n_tokens"].values >= 8)))     # a genuinely mixed review
print(neg.loc[i, "review"], "\n")
fig, ax = plt.subplots(figsize=(6, 2.2))
ax.barh([f"Topic {k}" for k in range(5)][::-1], doc_topic5[i][::-1], color=TOPIC_COLORS[:5][::-1])
ax.set_xlim(0, 1); ax.set_xlabel("P(topic | review)"); ax.set_title("One review = a mixture of topics")
plt.tight_layout(); plt.show()

### 4.2 A failure mode on purpose: generic words dominate  *(slide 24)*
What if we had skipped the domain and request stopwords? Same model, vocabulary built from the less-filtered tokens:

In [ ]:
loose_docs = neg["tokens_all"].map(lambda toks: " ".join(t for t in toks if t not in SW_GENERIC and len(t) > 2))   # generic stopwords only
cv_loose = CountVectorizer(token_pattern=r"\S+", min_df=5, max_df=0.90)
X_loose = cv_loose.fit_transform(loose_docs)
lda_loose = LatentDirichletAllocation(n_components=5, random_state=SEED, max_iter=10).fit(X_loose)
v_loose = cv_loose.get_feature_names_out()
display(pd.DataFrame({f"Topic {k}": v_loose[lda_loose.components_[k].argsort()[::-1][:8]] for k in range(5)}))

Brand, request and evaluative words (`aplikasi`, `brimo`, `nya`, `bagus`, `mohon`) appear at the top of several topics, so the topics repeat each other and are harder to tell apart. Most topic-model problems are fixed by **better vocabulary**, not by more iterations (slide 24).

### 4.3 How many topics?  *(slides 21 and 23)*
There is no magic K. We fit several values and look at three signals:
- **Perplexity** on held-out reviews (lower = better prediction). In scikit-learn it often gets *worse* as K grows and it does not track human judgement well, so treat it as a secondary signal.
- **Coherence (NPMI)**: do a topic's top words actually appear together in reviews? Higher is better (slide 23).
- **Topic diversity**: the share of unique words across all topics' top-10 lists. Low diversity means topics repeat each other.

In [ ]:
X_bin = (X > 0).astype(np.float32).tocsc()
doc_freq = np.asarray(X_bin.sum(axis=0)).ravel()

def npmi_coherence(model, top_n=10):
    """Mean NPMI over all pairs of each topic's top words, using document co-occurrence in our corpus."""
    scores = []
    for comp in model.components_:
        top = comp.argsort()[::-1][:top_n]
        joint = (X_bin[:, top].T @ X_bin[:, top]).toarray()
        vals = []
        for a, b in combinations(range(top_n), 2):
            p_ab = joint[a, b] / X.shape[0]
            if p_ab == 0:
                vals.append(-1.0); continue
            p_a, p_b = doc_freq[top[a]] / X.shape[0], doc_freq[top[b]] / X.shape[0]
            vals.append(np.log(p_ab / (p_a * p_b)) / -np.log(p_ab))
        scores.append(np.mean(vals))
    return float(np.mean(scores)), scores

def diversity(model, top_n=10):
    tops = [tuple(comp.argsort()[::-1][:top_n]) for comp in model.components_]
    return len(set(sum(map(list, tops), []))) / (top_n * len(tops))

perm = np.random.default_rng(SEED).permutation(X.shape[0])                 # random 85/15 split for held-out perplexity
X_tr, X_ho = X[perm[: int(0.85 * len(perm))]], X[perm[int(0.85 * len(perm)):]]
rows, models = [], {}
t0 = time.perf_counter()
for K in [3, 4, 5, 6, 8, 10, 12]:
    m = LatentDirichletAllocation(n_components=K, random_state=SEED, max_iter=15).fit(X_tr)
    models[K] = m
    rows.append({"K": K, "perplexity (held-out)": m.perplexity(X_ho), "coherence (NPMI)": npmi_coherence(m)[0],
                 "diversity": diversity(m)})
k_scan = pd.DataFrame(rows).set_index("K")
print(f"Scanned {len(k_scan)} values of K in {time.perf_counter() - t0:.0f}s")
display(k_scan.round(3))

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, col in zip(axes, k_scan.columns):
    ax.plot(k_scan.index, k_scan[col], marker="o", color="#2a78d6"); ax.set_title(col); ax.set_xlabel("K")
plt.tight_layout(); plt.show()

Our rule: among the values of K whose topics are still distinct (diversity ≥ 0.7), take the **smallest K whose coherence is within 0.01 of the best**: simpler is easier to explain and to monitor. Then check that a human can label every topic. If not, try a neighbouring K. Change `K_FINAL` if your reading of the topics disagrees with the numbers: **interpretability is part of the decision** (slide 21).

In [ ]:
eligible = k_scan[k_scan["diversity"] >= 0.7]
eligible = eligible if len(eligible) else k_scan
best_coh = eligible["coherence (NPMI)"].max()
K_FINAL = int(eligible.index[eligible["coherence (NPMI)"] >= best_coh - 0.01].min())
print("K_FINAL =", K_FINAL)

t0 = time.perf_counter()
lda = LatentDirichletAllocation(n_components=K_FINAL, random_state=SEED, learning_method="batch", max_iter=25).fit(X)
TIMINGS[f"LDA (K={K_FINAL})"] = time.perf_counter() - t0
doc_topic = lda.transform(X)

### 4.4 Is the result stable across random seeds?  *(slide 25: "try multiple seeds before trusting one result")*
We refit with three other seeds and match each topic to its most similar topic in the main model (Jaccard overlap of top-10 words, optimal matching).

In [ ]:
def top_sets(model, n=10):
    return [set(vocab[comp.argsort()[::-1][:n]]) for comp in model.components_]

base = top_sets(lda)
stab = []
for seed in [1, 2, 3]:
    other = top_sets(LatentDirichletAllocation(n_components=K_FINAL, random_state=seed, max_iter=15).fit(X))
    J = np.array([[len(a & b) / len(a | b) for b in other] for a in base])
    r, c = linear_sum_assignment(-J)
    stab.append(J[r, c])
stab = pd.DataFrame(stab, index=["seed 1", "seed 2", "seed 3"], columns=[f"Topic {k}" for k in range(K_FINAL)])
display(stab.round(2))
print("Mean matched overlap per topic:", stab.mean().round(2).to_dict())

Topics with high overlap across seeds are robust findings. Topics with low overlap change from run to run: report them with caution, or merge them.

### 4.5 Interpreting the topics  *(slide 22)*
Step 1: read 10–15 top words together. Step 2: read the most representative reviews. Step 3: write a stakeholder-readable label.

In [ ]:
weights = lda.components_ / lda.components_.sum(axis=1, keepdims=True)
coh_each = npmi_coherence(lda)[1]
for k in range(K_FINAL):
    words = vocab[weights[k].argsort()[::-1][:12]]
    print(f"── Topic {k}   coherence {coh_each[k]:.2f}   seed-stability {stab[f'Topic {k}'].mean():.2f}")
    print("   top words:", " • ".join(words))
    reps = np.argsort(-doc_topic[:, k] * (neg["n_tokens"].values >= 6))[:3]
    for j in reps:
        print(f"   [{doc_topic[j, k]:.2f}] {neg.loc[j, 'review'][:140]}")
    print()

In [ ]:
# Draft labels from the top-3 words. ✍️ Replace them with your own human-readable labels after reading the reviews above,
# e.g. {0: "Login & account access", 1: "Face/KTP verification", ...}
TOPIC_LABELS = {k: " / ".join(vocab[weights[k].argsort()[::-1][:3]]) for k in range(K_FINAL)}
TOPIC_LABELS.update({})          # ← put your labels here
for k, v in TOPIC_LABELS.items():
    print(k, "→", v)

| Weak label | Better label |
|---|---|
| "login" | Cannot log in / account access |
| "layar" | App does not open (blank white screen) |
| "verifikasi" | Face & KTP verification at registration |
| "saldo" | Money deducted but transaction failed |
| "jaringan" | Connection / network errors |

A topic label is a **human analytical judgment**, not a model output (slide 22).

### 4.6 Topic prevalence: explore, compare, monitor  *(slide 16)*
We now apply the model to **all** negative reviews (including duplicates) and track the share of each topic over time.

In [ ]:
neg_all = df[informative & (df["sentiment"] == "Negative")].copy()
theta = lda.transform(vectorizer.transform(neg_all["doc"]))
neg_all["topic"] = theta.argmax(axis=1)
neg_all["topic_weight"] = theta.max(axis=1)
neg_all["topic_label"] = neg_all["topic"].map(TOPIC_LABELS)

share = neg_all["topic_label"].value_counts(normalize=True)
monthly_share = pd.crosstab(neg_all["month"], neg_all["topic_label"], normalize="index")[share.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={"width_ratios": [1, 1.6]})
axes[0].barh(share.index[::-1], share.values[::-1], color=[TOPIC_COLORS[list(TOPIC_LABELS.values()).index(l)] for l in share.index[::-1]])
axes[0].set_xlabel("share of negative reviews"); axes[0].set_title("Topic prevalence (dominant topic)")
for lab in monthly_share.columns:
    axes[1].plot(monthly_share.index, monthly_share[lab], marker="o", lw=2, label=lab,
                 color=TOPIC_COLORS[list(TOPIC_LABELS.values()).index(lab)])
axes[1].set_title("Topic share per month (negative reviews)"); axes[1].legend(fontsize=7, loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout(); plt.show()

In [ ]:
# Compare: which topics are endorsed by other users (likes), and which come with 1 star?
cmp = neg_all.groupby("topic_label").agg(reviews=("review", "size"), likes=("likes", "sum"),
                                        one_star_share=("rating", lambda r: (r == 1).mean()))
cmp["share of reviews"] = cmp["reviews"] / cmp["reviews"].sum()
cmp["share of likes"] = cmp["likes"] / cmp["likes"].sum()
display(cmp.sort_values("share of reviews", ascending=False)[["share of reviews", "share of likes", "one_star_share"]]
        .style.format("{:.1%}"))

Prevalence by review count, by likes and by 1-star share can rank the same topics differently. A topic with a modest share of reviews but a large share of likes is one that **many other users recognise**, which is a strong case for prioritising it.

---
## 5. Semantic representations  *(slides 27–35)*
### 5.1 Word embeddings: words as dense vectors  *(slides 28–31)*
**Distributional idea:** words that occur in similar contexts get similar vectors. Word2Vec and GloVe are the famous implementations; here we build the same idea from scratch in a few lines, so every step is visible and nothing needs installing:

1. count how often each word appears near each other word (window of ±4 words),
2. turn counts into **PPMI** (positive pointwise mutual information: "more often together than chance"),
3. compress the sparse matrix to 100 dense dimensions with **SVD**.

We use a *light* normalisation here (no slang dictionary), so the embeddings can show us spelling variants we have not handled yet.

In [ ]:
def light_tokens(text):
    s = unicodedata.normalize("NFKC", str(text)).lower()
    return re.findall(r"[a-z]+", re.sub(r"([a-z])\1{2,}", r"\1", s))

t0 = time.perf_counter()
emb_docs = [light_tokens(t) for t in df["review"].drop_duplicates()]
wc = Counter(w for d in emb_docs for w in d)
emb_vocab = [w for w, c in wc.items() if c >= 15]
w_idx = {w: i for i, w in enumerate(emb_vocab)}
WINDOW = 4
rows, cols = [], []
for d in emb_docs:
    ids = [w_idx[w] for w in d if w in w_idx]
    for i, a in enumerate(ids):
        ctx = ids[max(0, i - WINDOW):i] + ids[i + 1:i + 1 + WINDOW]
        rows.extend([a] * len(ctx)); cols.extend(ctx)
V = len(emb_vocab)
C = sparse.coo_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)), shape=(V, V)).tocsr()
C.sum_duplicates()

total = C.sum()
row_sum = np.asarray(C.sum(axis=1)).ravel()
ctx_sum = np.asarray(C.sum(axis=0)).ravel() ** 0.75                  # context smoothing, as in word2vec
ctx_sum = ctx_sum / ctx_sum.sum() * total
Cc = C.tocoo()
pmi = np.log(Cc.data * total / (row_sum[Cc.row] * ctx_sum[Cc.col]))
keep = pmi > 0
PPMI = sparse.csr_matrix((pmi[keep], (Cc.row[keep], Cc.col[keep])), shape=(V, V))
W = normalize(TruncatedSVD(n_components=100, random_state=SEED).fit_transform(PPMI))
TIMINGS["Word embeddings (PPMI + SVD)"] = time.perf_counter() - t0
print(f"{V:,} words × 100 dimensions, built in {TIMINGS['Word embeddings (PPMI + SVD)']:.1f}s")
print("Vector for 'saldo' (first 8 of 100 dims):", W[w_idx["saldo"], :8].round(2))

**Cosine similarity** compares the *direction* of two vectors (slide 30): close to 1 = similar contexts, around 0 = unrelated. Because we normalised the vectors, cosine is just a dot product.

In [ ]:
def neighbours(word, k=8):
    sims = W @ W[w_idx[word]]
    return [(emb_vocab[j], round(float(sims[j]), 2)) for j in np.argsort(-sims)[1:k + 1]]

probe_words = ["lemot", "otp", "saldo", "error", "wajah", "jaringan", "cs", "mantap"]
display(pd.DataFrame({w: [f"{n} ({s})" for n, s in neighbours(w)] for w in probe_words if w in w_idx}))

Without any labels, the vectors learned that `lemot` ≈ `lelet` (slow), `otp` ≈ `kode`, `saldo` ≈ `berkurang`/`terpotong`, `error` ≈ `eror`, and that `cs` is close to `sabrina` (BRI's chatbot) and `chat`.

A practical use: neighbours that are **spelled almost the same** are often typos and variants. They are candidates for the slang dictionary in part 2 (this is how entries such as `tranfer`, `verivikasi` and `membatu` were found). Review them by hand: some are legitimate different words (`dibuka` vs `buka`, `bertransaksi` vs `transaksi`).

In [ ]:
from difflib import SequenceMatcher
cands = []
for w in [w for w, _ in Counter({w: wc[w] for w in emb_vocab if len(w) > 3}).most_common(400)]:
    for n, s in neighbours(w, 10):
        if s > 0.6 and SequenceMatcher(None, w, n).ratio() >= 0.8 and wc[n] < wc[w] and n not in SLANG:
            cands.append((n, w, s, wc[n]))
display(pd.DataFrame(cands, columns=["variant", "probably means", "cosine", "count"]).drop_duplicates("variant").head(15))

In [ ]:
groups = {"money": ["saldo", "uang", "transfer", "terpotong", "berkurang", "rekening"],
          "access": ["login", "masuk", "akun", "password", "pin", "username"],
          "verification": ["verifikasi", "wajah", "ktp", "foto", "selfie", "daftar"],
          "network": ["jaringan", "internet", "sinyal", "koneksi", "wifi", "data"],
          "praise": ["bagus", "mantap", "keren", "mudah", "cepat", "membantu"]}
words_2d = [(w, g) for g, ws in groups.items() for w in ws if w in w_idx]
xy = PCA(n_components=2, random_state=SEED).fit_transform(W[[w_idx[w] for w, _ in words_2d]])
fig, ax = plt.subplots(figsize=(7.5, 5.5))
for gi, g in enumerate(groups):
    pts = [(xy[i], w) for i, (w, gg) in enumerate(words_2d) if gg == g]
    ax.scatter([p[0][0] for p in pts], [p[0][1] for p in pts], color=TOPIC_COLORS[gi], s=40, label=g)
    for p, w in pts:
        ax.annotate(w, p, fontsize=8, xytext=(3, 3), textcoords="offset points")
ax.legend(); ax.set_title("Word vectors projected to 2D (PCA)"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

Similar meanings occupy nearby regions (slide 29). The 2D picture loses information (the real space has 100 dimensions), so use it for intuition, and use cosine similarity for decisions.

**Limitation: one word → one vector** (slide 31). Look at `bunga` (interest *or* flower):

In [ ]:
print("Neighbours of 'bunga':", neighbours("bunga") if "bunga" in w_idx else "not in vocabulary")

In a banking corpus `bunga` only appears in its finance sense (interest, instalments, KUR loans), so the vector is purely financial. A static embedding cannot represent "bunga mawar" (a rose) differently. Contextual models fix this (part 6).

### 5.2 Sentence embeddings: one vector per review  *(slides 32–33)*
We use **`paraphrase-multilingual-MiniLM-L12-v2`**, a Sentence-BERT model trained to place paraphrases close together, in 50+ languages including Indonesian. If it cannot be loaded, the notebook uses LSA (TF-IDF + SVD) instead and tells you.

In [ ]:
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
st_model = None
if ST_AVAILABLE:
    try:
        t0 = time.perf_counter()
        st_model = SentenceTransformer(MODEL_NAME, device=DEVICE)
        print(f"Loaded {MODEL_NAME} on {DEVICE} in {time.perf_counter() - t0:.0f}s")
    except Exception as e:
        print("Could not load the model:", type(e).__name__, str(e)[:120])

if st_model is None:
    lsa = make_pipeline(TfidfVectorizer(min_df=3, ngram_range=(1, 2), sublinear_tf=True),
                        TruncatedSVD(n_components=256, random_state=SEED), Normalizer())
    lsa.fit(df["review"].str.lower().drop_duplicates())
    print("Using the LSA fallback (TF-IDF + SVD). Results in 5.2–6 will be weaker than with a sentence-transformer.")
EMBEDDER = "sentence-transformer" if st_model is not None else "LSA fallback"

def embed(texts, batch_size=64):
    texts = [str(t) for t in texts]
    if st_model is not None:
        return st_model.encode(texts, batch_size=batch_size, show_progress_bar=len(texts) > 1000,
                               normalize_embeddings=True, convert_to_numpy=True)
    return lsa.transform([t.lower() for t in texts])

In [ ]:
E_pairs = embed([t for p in PAIRS for t in p])
lexical[f"embedding cosine ({EMBEDDER})"] = [float(E_pairs[2 * i] @ E_pairs[2 * i + 1]) for i in range(len(PAIRS))]
display(lexical.round(2))

With a sentence-transformer, the three paraphrase pairs get high similarity and the unrelated pair a low one, even though they share almost no words (slide 33). Note that "high" is relative: compare scores *within* one model, and check a few examples before choosing a threshold (slide 30: *similarity is model- and domain-dependent*).

### 5.3 Semantic search over complaints
Encode every unique informative negative review once, then search by meaning.

In [ ]:
t0 = time.perf_counter()
E_neg = embed(neg["review"].tolist())
TIMINGS[f"Sentence embeddings ({EMBEDDER})"] = time.perf_counter() - t0
print(f"Encoded {len(E_neg):,} reviews → {E_neg.shape[1]} dimensions in {TIMINGS[f'Sentence embeddings ({EMBEDDER})']:.0f}s")

def semantic_search(query, k=5):
    sims = E_neg @ embed([query])[0]
    top = np.argsort(-sims)[:k]
    return pd.DataFrame({"similarity": sims[top].round(2), "review": neg["review"].iloc[top].str[:130].values})

def keyword_search(query):
    words = set(to_content(normalize_tokens(query)))
    return int(neg["tokens"].map(lambda t: words <= set(t)).sum())

for q in ["uang sudah terpotong tapi transaksinya gagal",
          "tidak bisa masuk karena lupa kata sandi",
          "aplikasi sering keluar sendiri waktu dipakai"]:
    print(f"Query: {q!r}   (reviews containing all its keywords: {keyword_search(q)})")
    display(semantic_search(q))

Keyword search only finds reviews that use the exact words; semantic search also finds "saldo berkurang tapi tidak masuk" or "force close". This is how a customer-service team can find **all** reports of an issue, whatever words people use.

### 5.4 From embeddings to semantic clusters  *(slide 35)*
Once every review is a vector, k-means can group reviews by meaning. We describe each cluster with **class-based TF-IDF** (words frequent in this cluster but not in the others) and the reviews closest to the cluster centre.

In [ ]:
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(E_neg), size=min(4000, len(E_neg)), replace=False)
sil = {}
for k in [4, 6, 8, 10]:
    labels_k = KMeans(n_clusters=k, n_init=5, random_state=SEED).fit_predict(E_neg[sample_idx])
    sil[k] = silhouette_score(E_neg[sample_idx], labels_k, metric="cosine")
print("Silhouette (higher = better separated):", {k: round(v, 3) for k, v in sil.items()})

N_CLUSTERS = K_FINAL                     # same number as the LDA topics, so we can compare them
km = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=SEED).fit(E_neg)
neg["cluster"] = km.labels_

In [ ]:
def class_tfidf(doc_series, labels, top_n=10):
    """c-TF-IDF: treat each cluster as one big document (as in BERTopic)."""
    joined = doc_series.groupby(labels).agg(" ".join)
    cv = CountVectorizer(token_pattern=r"\S+", min_df=1)
    tf = cv.fit_transform(joined).toarray().astype(float)
    tf_norm = tf / tf.sum(axis=1, keepdims=True)
    idf = np.log(1 + tf.sum(axis=1).mean() / tf.sum(axis=0))
    scores = tf_norm * idf
    terms_ = cv.get_feature_names_out()
    return {c: list(terms_[scores[i].argsort()[::-1][:top_n]]) for i, c in enumerate(joined.index)}

cluster_words = class_tfidf(neg["doc"], neg["cluster"])
for c in range(N_CLUSTERS):
    members = np.where(neg["cluster"].values == c)[0]
    centre_dist = E_neg[members] @ km.cluster_centers_[c]
    print(f"── Cluster {c}  ({len(members):,} reviews, {len(members) / len(neg):.0%})")
    print("   words:", " • ".join(cluster_words[c]))
    for j in members[np.argsort(-centre_dist)[:3]]:
        print("   -", neg["review"].iloc[j][:130])
    print()

In [ ]:
neg["lda_topic"] = doc_topic.argmax(axis=1)
ct = pd.crosstab(neg["cluster"], neg["lda_topic"].map(TOPIC_LABELS), normalize="index")
fig, ax = plt.subplots(figsize=(9, 3.8))
im = ax.imshow(ct.values, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(ct.shape[1]), ct.columns, rotation=30, ha="right", fontsize=8)
ax.set_yticks(range(ct.shape[0]), [f"cluster {c}" for c in ct.index])
for (r, c), v in np.ndenumerate(ct.values):
    ax.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=7, color="white" if v > 0.5 else "black")
ax.set_title("Where do embedding clusters fall among LDA topics? (row share)")
plt.colorbar(im, ax=ax, fraction=0.03); plt.tight_layout(); plt.show()
print(f"Adjusted mutual information between clusters and LDA topics: "
      f"{adjusted_mutual_info_score(neg['cluster'], neg['lda_topic']):.2f}  (1 = identical grouping, 0 = unrelated)")

The two methods partly agree: some clusters map almost entirely onto one LDA topic, others split or merge themes. Neither is "correct". LDA groups by **shared words**; embedding clusters group by **meaning**, so paraphrases with different vocabulary can land together. Clustering still needs human interpretation and a decision about the number of clusters (slide 35).

---
## 6. Transformer-based representations  *(slides 37–39)*
### 6.1 Contextual vectors: the same word, different meanings
A transformer encoder gives every token a vector that depends on the **whole sentence** (self-attention, slide 37). We take the token vectors for `bunga` from the same multilingual model in four sentences: two about interest (finance), two about flowers.

In [ ]:
CONTEXT_SENTENCES = ["bunga deposito di bank ini naik bulan lalu",
                     "bunga pinjaman kur terlalu tinggi untuk usaha kecil",
                     "saya membeli bunga mawar merah untuk ibu",
                     "taman itu penuh bunga warna warni yang harum"]

def word_vector_in_context(sentence, word):
    enc = st_model.tokenizer(sentence, return_offsets_mapping=True, truncation=True)
    tok_emb = st_model.encode([sentence], output_value="token_embeddings")[0]
    tok_emb = tok_emb.detach().cpu().numpy() if hasattr(tok_emb, "detach") else np.asarray(tok_emb)
    start = sentence.index(word); end = start + len(word)
    idx = [i for i, (a, b) in enumerate(enc["offset_mapping"]) if b > a and a < end and b > start]
    vec = tok_emb[idx].mean(axis=0)
    return vec / np.linalg.norm(vec)

if st_model is not None:
    pieces = st_model.tokenizer.tokenize(CONTEXT_SENTENCES[0])
    print("How the model tokenises sentence 1:", pieces)
    vecs = np.vstack([word_vector_in_context(s, "bunga") for s in CONTEXT_SENTENCES])
    ctx_sim = pd.DataFrame(vecs @ vecs.T, index=[s[:32] + "…" for s in CONTEXT_SENTENCES],
                           columns=["fin 1", "fin 2", "flower 1", "flower 2"])
    display(ctx_sim.round(2).style.background_gradient(cmap="Blues", vmin=0.5, vmax=1))
else:
    print("Skipped: needs the sentence-transformer model (see 5.2).")
print("Static embedding (5.1): 'bunga' has exactly one vector, so all four similarities would be 1.00.")

The finance–finance and flower–flower pairs are more similar to each other than finance–flower pairs: the vector for `bunga` changes with its context (slide 38). The static PPMI vector cannot do this.

### 6.2 Two practical notes
- **Subword tokens.** Transformers split rare words into pieces (`▁bung`, `a` …). To get one vector per word we average its pieces, as above.
- **Sentence vectors.** A raw BERT `[CLS]` vector is *not* automatically a good sentence-similarity representation (slide 39). Models such as the one we used are trained specifically for similarity, which is why 5.2–5.4 work.

---
## 7. Bringing it together  *(slides 40–41)*
### 7.1 Which representation should we use?

In [ ]:
rep_table = pd.DataFrame([
    ["Term frequency", "Strong", "Weak", "Weak", "Very high", "Low"],
    ["TF-IDF", "Strong", "Weak", "Weak", "High", "Low"],
    ["LDA", "Medium", "Strong", "Limited", "Medium–High", "Medium"],
    ["Word embeddings", "Medium", "Indirect", "Strong (word-level)", "Medium", "Medium"],
    ["Sentence embeddings", "Low–Medium", "Indirect", "Strong (text-level)", "Medium–Low", "Medium"],
    ["Transformer task model", "Low–Medium", "Depends on task", "Strong/contextual", "Lower", "Higher"],
], columns=["Method", "Exact terms", "Themes", "Semantic similarity", "Interpretability", "Cost"]).set_index("Method")
display(rep_table)
print("Measured in this notebook:")
for k, v in TIMINGS.items():
    print(f"  {k:<45} {v:6.1f}s")

**Not every problem needs the most complex model.** For a weekly dashboard of "what are people complaining about", TF-IDF keywords plus LDA may be enough and fully explainable. For finding *every* report of one incident, whatever words users chose, sentence embeddings are worth their cost.

### 7.2 From sentiment to actionable themes  *(slide 41)*
A useful answer combines **polarity**, **themes** and **representative examples**:

In [ ]:
neg_share = (df["sentiment"] == "Negative").mean()
counts_m = neg_all.groupby("month").size()
full_months = monthly_share.index[counts_m.reindex(monthly_share.index).values >= 0.6 * counts_m.median()]
first3, last3 = full_months[:3], full_months[-3:]

rows = []
for lab in share.index:
    sub = neg_all[(neg_all["topic_label"] == lab) & (neg_all["n_tokens"] >= 6) & (neg_all["topic_weight"] >= 0.6)]
    liked = sub[sub["likes"] > 0]
    pick = liked if len(liked) else sub
    example = pick.sort_values(["likes", "topic_weight"], ascending=False)["review"].iloc[0] if len(pick) else ""
    trend = monthly_share.loc[last3, lab].mean() - monthly_share.loc[first3, lab].mean()
    rows.append({"theme": lab, "share of negative": share[lab], "trend (pp)": trend * 100,
                 "example (clearly on-topic, most liked)": example[:110], "action (to be decided with the stakeholder)": ""})
themes = pd.DataFrame(rows).set_index("theme")
display(themes.style.format({"share of negative": "{:.0%}", "trend (pp)": "{:+.1f}"}))

top3 = share.head(3)
print(f"SIGNAL: {neg_share:.0%} of BRImo reviews are negative.")
print("EXPLANATION: negative reviews cluster around " +
      ", ".join(f"{lab} ({s:.0%})" for lab, s in top3.items()) + ".")
print(f"(trend = change in share between {first3[0]:%b %Y}–{first3[-1]:%b %Y} and {last3[0]:%b %Y}–{last3[-1]:%b %Y}; "
      "partial months excluded)")

Compare this with "26% of reviews are negative" (slide 3). The table tells the bank **what** to fix, **how big** each issue is, **whether it is growing**, and gives a **real quote** to show to the team that owns it. The action column is deliberately empty: deciding actions is the analyst's job together with the stakeholder.

---
## 8. Class activity
**A. Label and validate the topics (15 min).** In pairs, edit `TOPIC_LABELS` in 4.5 with stakeholder-readable labels. Then read 10 random reviews of each topic with the cell below. How many fit the label? Would you merge or split any topic?

In [ ]:
TOPIC_TO_CHECK = 0          # ✍️ change me
check_reviews = neg_all[neg_all["topic"] == TOPIC_TO_CHECK].sample(10, random_state=1)["review"]
print(TOPIC_LABELS[TOPIC_TO_CHECK])
for r in check_reviews:
    print(" -", r[:150])

**B. What do users praise? (15 min).** Run the same topic pipeline on **positive** reviews. Which K works? Are the topics as clear as for negative reviews, and why (hint: look at review length)?

In [ ]:
SEGMENT, K_SEGMENT = "Positive", 4          # ✍️ try "Neutral" too
seg = df[informative & (df["sentiment"] == SEGMENT)].drop_duplicates("doc")
cv_seg = CountVectorizer(token_pattern=r"\S+", min_df=5, max_df=0.90)
X_seg = cv_seg.fit_transform(seg["doc"])
lda_seg = LatentDirichletAllocation(n_components=K_SEGMENT, random_state=SEED, max_iter=15).fit(X_seg)
v_seg = cv_seg.get_feature_names_out()
print(f"{SEGMENT}: {X_seg.shape[0]:,} unique informative reviews")
display(pd.DataFrame({f"Topic {k}": v_seg[lda_seg.components_[k].argsort()[::-1][:10]] for k in range(K_SEGMENT)}))

**C. Semantic search for a stakeholder (10 min).** Write three queries a BRI customer-service manager would ask (for example about QRIS payments, blocked accounts or the chatbot) and run them with `semantic_search`. Are the results relevant? Where does keyword search fail?

In [ ]:
for q in ["pembayaran qris gagal tapi saldo terpotong"]:      # ✍️ add your queries
    display(semantic_search(q, k=5))